# RKD Multi-Dataset Runner (Google Colab Ready)

This notebook is self-contained for Colab and reproduces the three scripts:
- `examples/cub200.sh`
- `examples/cars196.sh`
- `examples/stanford.sh`

For each dataset, it runs:
1. Teacher training (`run.py`)
2. Student distillation with distance + angle (`run_distill.py`)
3. Student distillation with quadruplet-only (`run_distill.py`, `quad_ratio=1`)
4. Self-distillation with distance + angle
5. Self-distillation with quadruplet-only

Run cells from top to bottom.

In [ ]:
import os
import shlex
import subprocess
import sys
from pathlib import Path

USE_DRIVE = False  # Set True to persist checkpoints in Google Drive.
REPO_URL = "https://github.com/Gabomfim/MO434.git"


def run_cmd(cmd):
    cmd = [str(x) for x in cmd]
    print("\n>>>", " ".join(shlex.quote(x) for x in cmd))
    subprocess.run(cmd, check=True)


if USE_DRIVE:
    try:
        from google.colab import drive
    except ImportError as exc:
        raise RuntimeError("USE_DRIVE=True requires Google Colab.") from exc
    drive.mount("/content/drive")
    base_root = Path("/content/drive/MyDrive/mo434")
else:
    base_root = Path("/content")

base_root.mkdir(parents=True, exist_ok=True)
repo_root = base_root / "MO434"
rkd_root = repo_root / "RKD"

if not repo_root.exists():
    run_cmd(["git", "clone", REPO_URL, str(repo_root)])
else:
    run_cmd(["git", "-C", str(repo_root), "pull", "--ff-only"])

os.chdir(rkd_root)
print("Working directory:", os.getcwd())

run_cmd(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "tqdm",
        "h5py",
        "scipy",
        "wandb",
        "kagglehub",
    ]
)

data_root = repo_root / "data"
data_root.mkdir(parents=True, exist_ok=True)

checkpoint_root = repo_root / "checkpoints_colab"
checkpoint_root.mkdir(parents=True, exist_ok=True)

print("Data root:", data_root)
print("Checkpoint root:", checkpoint_root)

In [ ]:
import torch
import run as teacher_runner
import run_distill as distill_runner

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU is required. In Colab, go to Runtime > Change runtime type > GPU."
    )

print("GPU:", torch.cuda.get_device_name(0))
print("CUDA version:", torch.version.cuda)

WANDB_PROJECT = "rkd-metric-learning"
WANDB_ENTITY = ""
WANDB_MODE = "online"  # online | offline | disabled

RUN_DATASETS = ["cub200", "cars196", "stanford"]

DATASET_CONFIGS = {
    "cub200": {
        "teacher": {
            "epochs": 40,
            "lr_decay_epochs": [25, 30, 35],
            "lr_decay_gamma": 0.5,
            "batch": 128,
            "embedding_size": 512,
            "save_dir": "cub200_resnet50_512",
        },
        "distill": {
            "epochs": 80,
            "lr_decay_epochs": [40, 60],
            "lr_decay_gamma": 0.1,
            "batch": 128,
            "iter_per_epoch": 100,
            "student_small_embedding": 128,
        },
    },
    "cars196": {
        "teacher": {
            "epochs": 40,
            "lr_decay_epochs": [25, 30, 35],
            "lr_decay_gamma": 0.5,
            "batch": 128,
            "embedding_size": 512,
            "save_dir": "cars196_resnet50_512",
        },
        "distill": {
            "epochs": 80,
            "lr_decay_epochs": [40, 60],
            "lr_decay_gamma": 0.1,
            "batch": 128,
            "iter_per_epoch": 100,
            "student_small_embedding": 128,
        },
    },
    "stanford": {
        "teacher": {
            "epochs": 40,
            "lr_decay_epochs": [25, 30, 35],
            "lr_decay_gamma": 0.5,
            "batch": 128,
            "iter_per_epoch": 1000,
            "embedding_size": 512,
            "save_dir": "stanford_resnet50_512",
        },
        "distill": {
            "epochs": 120,
            "lr_decay_epochs": [40, 80],
            "lr_decay_gamma": 0.1,
            "batch": 128,
            "iter_per_epoch": 500,
            "student_small_embedding": 64,
        },
    },
}

In [ ]:
import kagglehub


def resolve_data_dir(dataset_name):
    if dataset_name == "cub200":
        path = Path(kagglehub.dataset_download("wenewone/cub2002011"))
        data_dir = path.parent if path.name == "CUB_200_2011" else path
        print(f"Using CUB Kaggle data root: {data_dir}")
        return str(data_dir)

    print(f"Using shared data root for {dataset_name}: {data_root}")
    return str(data_root)


def _inject_wandb(params, run_name, group, job_type, tags):
    params["wandb_project"] = WANDB_PROJECT
    params["wandb_run_name"] = run_name
    params["wandb_group"] = group
    params["wandb_job_type"] = job_type
    params["wandb_tags"] = tags
    params["wandb_mode"] = WANDB_MODE
    if WANDB_ENTITY.strip():
        params["wandb_entity"] = WANDB_ENTITY


def _ckpt_path(*parts):
    return str(Path(checkpoint_root, *parts))


def run_dataset(dataset_name):
    cfg = DATASET_CONFIGS[dataset_name]
    t_cfg = cfg["teacher"]
    d_cfg = cfg["distill"]
    dataset_data_dir = resolve_data_dir(dataset_name)

    teacher_dir = _ckpt_path(t_cfg["save_dir"])
    teacher_ckpt = str(Path(teacher_dir) / "best.pth")

    print(f"===== [{dataset_name}] Teacher =====")
    teacher_params = {
        "mode": "train",
        "dataset": dataset_name,
        "base": "resnet50",
        "sample": "distance",
        "loss": "l2_triplet",
        "margin": 0.2,
        "embedding_size": t_cfg["embedding_size"],
        "epochs": t_cfg["epochs"],
        "lr_decay_epochs": t_cfg["lr_decay_epochs"],
        "lr_decay_gamma": t_cfg["lr_decay_gamma"],
        "batch": t_cfg["batch"],
        "data": dataset_data_dir,
        "save_dir": teacher_dir,
    }
    if "iter_per_epoch" in t_cfg:
        teacher_params["iter_per_epoch"] = t_cfg["iter_per_epoch"]
    _inject_wandb(
        teacher_params,
        run_name=f"{dataset_name}-teacher-resnet50-512",
        group=f"{dataset_name}-teacher",
        job_type="teacher",
        tags=[dataset_name, "teacher", "resnet50", "baseline"],
    )
    teacher_runner.run_with_params(teacher_params)

    print(f"===== [{dataset_name}] Student (resnet18, dist+angle) =====")
    student_da_dir = _ckpt_path(
        f"{dataset_name}_student_resnet18_{d_cfg['student_small_embedding']}"
    )
    student_da_params = {
        "dataset": dataset_name,
        "base": "resnet18",
        "embedding_size": d_cfg["student_small_embedding"],
        "l2normalize": "false",
        "dist_ratio": 1,
        "angle_ratio": 2,
        "teacher_base": "resnet50",
        "teacher_embedding_size": 512,
        "teacher_load": teacher_ckpt,
        "epochs": d_cfg["epochs"],
        "iter_per_epoch": d_cfg["iter_per_epoch"],
        "lr_decay_epochs": d_cfg["lr_decay_epochs"],
        "lr_decay_gamma": d_cfg["lr_decay_gamma"],
        "batch": d_cfg["batch"],
        "data": dataset_data_dir,
        "save_dir": student_da_dir,
    }
    _inject_wandb(
        student_da_params,
        run_name=f"{dataset_name}-student-resnet18-dist-angle",
        group=f"{dataset_name}-student-r18-da",
        job_type="distillation",
        tags=[dataset_name, "student", "resnet18", "dist-angle"],
    )
    distill_runner.run_with_params(student_da_params)

    print(f"===== [{dataset_name}] Student (resnet18, quadruplet-only) =====")
    student_quad_dir = _ckpt_path(
        f"{dataset_name}_student_resnet18_{d_cfg['student_small_embedding']}_quad"
    )
    student_quad_params = {
        "dataset": dataset_name,
        "base": "resnet18",
        "embedding_size": d_cfg["student_small_embedding"],
        "l2normalize": "false",
        "quad_ratio": 1,
        "teacher_base": "resnet50",
        "teacher_embedding_size": 512,
        "teacher_load": teacher_ckpt,
        "epochs": d_cfg["epochs"],
        "iter_per_epoch": d_cfg["iter_per_epoch"],
        "lr_decay_epochs": d_cfg["lr_decay_epochs"],
        "lr_decay_gamma": d_cfg["lr_decay_gamma"],
        "batch": d_cfg["batch"],
        "data": dataset_data_dir,
        "save_dir": student_quad_dir,
    }
    _inject_wandb(
        student_quad_params,
        run_name=f"{dataset_name}-student-resnet18-quad",
        group=f"{dataset_name}-student-r18-quad",
        job_type="distillation",
        tags=[dataset_name, "student", "resnet18", "quad-only"],
    )
    distill_runner.run_with_params(student_quad_params)

    print(f"===== [{dataset_name}] Self-distill (resnet50, dist+angle) =====")
    self_da_dir = _ckpt_path(f"{dataset_name}_student_resnet50_512")
    self_da_params = {
        "dataset": dataset_name,
        "base": "resnet50",
        "embedding_size": 512,
        "l2normalize": "false",
        "dist_ratio": 1,
        "angle_ratio": 2,
        "teacher_base": "resnet50",
        "teacher_embedding_size": 512,
        "teacher_load": teacher_ckpt,
        "epochs": d_cfg["epochs"],
        "iter_per_epoch": d_cfg["iter_per_epoch"],
        "lr_decay_epochs": d_cfg["lr_decay_epochs"],
        "lr_decay_gamma": d_cfg["lr_decay_gamma"],
        "batch": d_cfg["batch"],
        "data": dataset_data_dir,
        "save_dir": self_da_dir,
    }
    _inject_wandb(
        self_da_params,
        run_name=f"{dataset_name}-student-resnet50-dist-angle",
        group=f"{dataset_name}-self-r50-da",
        job_type="self-distillation",
        tags=[dataset_name, "self-distill", "resnet50", "dist-angle"],
    )
    distill_runner.run_with_params(self_da_params)

    print(f"===== [{dataset_name}] Self-distill (resnet50, quadruplet-only) =====")
    self_quad_dir = _ckpt_path(f"{dataset_name}_student_resnet50_512_quad")
    self_quad_params = {
        "dataset": dataset_name,
        "base": "resnet50",
        "embedding_size": 512,
        "l2normalize": "false",
        "quad_ratio": 1,
        "teacher_base": "resnet50",
        "teacher_embedding_size": 512,
        "teacher_load": teacher_ckpt,
        "epochs": d_cfg["epochs"],
        "iter_per_epoch": d_cfg["iter_per_epoch"],
        "lr_decay_epochs": d_cfg["lr_decay_epochs"],
        "lr_decay_gamma": d_cfg["lr_decay_gamma"],
        "batch": d_cfg["batch"],
        "data": dataset_data_dir,
        "save_dir": self_quad_dir,
    }
    _inject_wandb(
        self_quad_params,
        run_name=f"{dataset_name}-student-resnet50-quad",
        group=f"{dataset_name}-self-r50-quad",
        job_type="self-distillation",
        tags=[dataset_name, "self-distill", "resnet50", "quad-only"],
    )
    distill_runner.run_with_params(self_quad_params)

In [ ]:
# Runs all experiments in the same order as the three example scripts.
for dataset_name in RUN_DATASETS:
    run_dataset(dataset_name)